# Assignment 4 - Delta Lake Essentials

This notebook is a full solution for the Delta Lake assignment covering Delta table creation, ACID operations, merge, schema evolution, time travel, and validation.


## Objective

Build and maintain a customer transactions table using Delta Lake features in Databricks.


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp
from delta.tables import DeltaTable
spark = SparkSession.builder.getOrCreate()
table_name = "workspace.default.assignment9_delta_demo"


## Phase 1 - Create Initial Delta Table


In [2]:
data = [
    (1, "C001", "Laptop", 50000),
    (2, "C002", "Mobile", 15000),
    (3, "C003", "Tablet", 20000),
    (4, "C004", "Laptop", 55000)
]
columns = ["id", "customer_id", "product", "amount"]
df = spark.createDataFrame(data, columns)
df.show()
spark.sql("DROP TABLE IF EXISTS workspace.default.assignment9_delta_demo")
df.write.format("delta").mode("overwrite").saveAsTable(table_name)
delta_df = spark.table(table_name)
print("Initial Delta table")
delta_df.show()


+---+-----------+-------+------+
| id|customer_id|product|amount|
+---+-----------+-------+------+
|  1|       C001| Laptop| 50000|
|  2|       C002| Mobile| 15000|
|  3|       C003| Tablet| 20000|
|  4|       C004| Laptop| 55000|
+---+-----------+-------+------+

Initial Delta table
+---+-----------+-------+------+
| id|customer_id|product|amount|
+---+-----------+-------+------+
|  1|       C001| Laptop| 50000|
|  2|       C002| Mobile| 15000|
|  3|       C003| Tablet| 20000|
|  4|       C004| Laptop| 55000|
+---+-----------+-------+------+



## Phase 2 - Insert, Update, Delete


In [3]:
new_rows = [(5, "C005", "Camera", 30000)]
new_df = spark.createDataFrame(new_rows, columns)
new_df.write.mode("append").insertInto(table_name)
delta_table = DeltaTable.forName(spark, table_name)
delta_table.update(condition = col("id") == 2, set = {"amount": lit(18000)})
delta_table.delete(condition = col("id") == 1)
print("After insert, update, delete")
spark.table(table_name).orderBy("id").show()


After insert, update, delete
+---+-----------+-------+------+
| id|customer_id|product|amount|
+---+-----------+-------+------+
|  2|       C002| Mobile| 18000|
|  3|       C003| Tablet| 20000|
|  4|       C004| Laptop| 55000|
|  5|       C005| Camera| 30000|
+---+-----------+-------+------+



## Phase 3 - MERGE INTO for Incremental Load


In [4]:
updates = [
    (3, "C003", "Tablet", 22000),
    (6, "C006", "Watch", 8000)
]
updates_df = spark.createDataFrame(updates, columns)
print("Before MERGE count:", spark.table(table_name).count())
delta_table.alias("target").merge(
    updates_df.alias("source"),
    "target.id = source.id"
).whenMatchedUpdate(set = {
    "customer_id": col("source.customer_id"),
    "product": col("source.product"),
    "amount": col("source.amount")
}).whenNotMatchedInsert(values = {
    "id": col("source.id"),
    "customer_id": col("source.customer_id"),
    "product": col("source.product"),
    "amount": col("source.amount")
}).execute()
merged_df = spark.table(table_name)
print("After MERGE count:", merged_df.count())
merged_df.orderBy("id").show()


Before MERGE count: 4
After MERGE count: 5
+---+-----------+-------+------+
| id|customer_id|product|amount|
+---+-----------+-------+------+
|  2|       C002| Mobile| 18000|
|  3|       C003| Tablet| 22000|
|  4|       C004| Laptop| 55000|
|  5|       C005| Camera| 30000|
|  6|       C006|  Watch|  8000|
+---+-----------+-------+------+



## Phase 4 - Schema Evolution


In [5]:
updates_with_category = [
    (3, "C003", "Tablet", 23000, "Electronics"),
    (7, "C007", "Headphones", 5000, "Accessories")
]
columns_with_category = ["id", "customer_id", "product", "amount", "category"]
updates_schema_df = spark.createDataFrame(updates_with_category, columns_with_category)
updates_schema_df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(table_name)
evolved_df = spark.table(table_name)
print("Schema after evolution")
evolved_df.printSchema()
evolved_df.orderBy("id").show()


Schema after evolution
root
 |-- id: long (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: long (nullable = true)
 |-- category: string (nullable = true)

+---+-----------+----------+------+-----------+
| id|customer_id|   product|amount|   category|
+---+-----------+----------+------+-----------+
|  2|       C002|    Mobile| 18000|       NULL|
|  3|       C003|    Tablet| 22000|       NULL|
|  3|       C003|    Tablet| 23000|Electronics|
|  4|       C004|    Laptop| 55000|       NULL|
|  5|       C005|    Camera| 30000|       NULL|
|  6|       C006|     Watch|  8000|       NULL|
|  7|       C007|Headphones|  5000|Accessories|
+---+-----------+----------+------+-----------+



## Phase 5 - Time Travel and History


In [6]:
print("History")
spark.sql(f"DESCRIBE HISTORY {table_name}").show(truncate=False)
print("Current version data")
spark.table(table_name).show()
print("Example time travel to version 0")
spark.read.option("versionAsOf", 0).table(table_name).show()


History
+-------+-------------------+--------------+------------------------+---------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------+------------------+------------------------------------+------------------------+-----------+-----------------+-------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Optional Restore Example


In [7]:
%sql
-- Run this in Databricks SQL if you want to restore the table to an old version
-- RESTORE TABLE workspace.default.assignment9_delta_demo TO VERSION AS OF 0;


## Phase 6 - Validation


In [8]:
final_df = spark.table(table_name)
print("Total row count:", final_df.count())
print("Duplicate ids:")
final_df.groupBy("id").count().filter(col("count") > 1).show()
print("Updated record for id = 3")
final_df.filter(col("id") == 3).show()
print("Check for negative amounts")
final_df.filter(col("amount") < 0).show()
print("Final dataset")
final_df.orderBy("id").show()


Total row count: 7
Duplicate ids:
+---+-----+
| id|count|
+---+-----+
|  3|    2|
+---+-----+

Updated record for id = 3
+---+-----------+-------+------+-----------+
| id|customer_id|product|amount|   category|
+---+-----------+-------+------+-----------+
|  3|       C003| Tablet| 23000|Electronics|
|  3|       C003| Tablet| 22000|       NULL|
+---+-----------+-------+------+-----------+

Check for negative amounts
+---+-----------+-------+------+--------+
| id|customer_id|product|amount|category|
+---+-----------+-------+------+--------+
+---+-----------+-------+------+--------+

Final dataset
+---+-----------+----------+------+-----------+
| id|customer_id|   product|amount|   category|
+---+-----------+----------+------+-----------+
|  2|       C002|    Mobile| 18000|       NULL|
|  3|       C003|    Tablet| 22000|       NULL|
|  3|       C003|    Tablet| 23000|Electronics|
|  4|       C004|    Laptop| 55000|       NULL|
|  5|       C005|    Camera| 30000|       NULL|
|  6|       C0

## Key Learning Points

- Delta Lake supports ACID-style updates on data lake storage.
- `MERGE` helps combine insert and update logic for incremental loads.
- Schema evolution helps absorb new columns safely.
- Time travel and history make rollback and auditing possible.
